# 시계열 검증과 정보 누수

## 철강산업 전력사용량의 과거–미래 분할

예측 시점에 이용할 수 있는 정보의 경계를 정하고, random split과 chronological split이 평가 결과를 어떻게 바꾸는지 비교합니다.


## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- forecast origin과 예측 시점에 이용 가능한 feature를 정의합니다.
- random split과 chronological holdout의 차이를 설명합니다.
- lag feature와 persistence baseline을 만들고 같은 미래 기간에서 후보모형과 비교합니다.


## 이번 교시에서 사용할 데이터

### 강의 시연: Steel 15분 시계열

UCI Steel 자료의 35,040개 관측을 `date` 순서로 정렬합니다. 한 행은 15분 구간이며, 직전 15분 사용량을 정확한 timestamp join으로 연결해 one-step-ahead feature를 만듭니다.

- **예측 대상:** 다음 15분의 `Usage_kWh`
- **시간 feature:** 예측 시점 이전에 알려진 lag와 달력정보
- **비교할 split:** random split과 과거 학습–미래 평가 split
- **baseline:** 직전 관측을 그대로 사용하는 persistence

### 적용 실습: Bike 시간별 시계열

17,379개의 시간별 대여 관측을 날짜와 시간으로 결합해 timestamp를 만듭니다. 정확히 한 시간 전 `cnt`가 존재할 때만 lag를 연결하고, 앞 기간으로 학습해 뒤 기간을 평가합니다. 관측이 빠진 시간에는 단순 행 이동이 정확한 1시간 lag가 아닐 수 있으므로 timestamp 차이를 확인합니다.


## Forecast origin: 예측에 사용할 수 있는 정보의 경계

forecast origin은 예측을 생성하는 기준 시점입니다. 이 시점까지 확보된 정보만
feature 생성, 전처리와 모형 적합에 사용할 수 있습니다.

$$
\underbrace{\text{이미 관측된 과거}}_{\text{사용 가능}}
\;\Big|\;\text{forecast origin}\;\Big|\;
\underbrace{\text{이후에 관측될 값}}_{\text{평가용 정답}}
$$

결과에는 origin, horizon, train 종료시각과 test 시작시각을 함께 기록합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

timeline = pd.date_range("2026-01-01 08:00", periods=5, freq="h")
origin = pd.Timestamp("2026-01-01 10:00")
view = pd.DataFrame({"timestamp": timeline})
view["status_at_origin"] = view["timestamp"].le(origin).map({True: "known", False: "future target"})

print("[시간경계로 확인] forecast origin 이전과 이후의 역할이 다릅니다.")
print("forecast origin:", origin)
display(view)

fig, ax = plt.subplots(figsize=(8, 2.7))
colors = view["status_at_origin"].map({"known": "#2563eb", "future target": "#dc2626"})
ax.scatter(view["timestamp"], [1] * len(view), c=colors, s=75)
ax.axvline(origin, color="black", linestyle="--", label="forecast origin")
ax.set(yticks=[], xlabel="time", title="Only information available by the origin can be used")
ax.legend()
plt.tight_layout()
plt.show()


## Exact lag: 앞 행이 아니라 정확한 과거시각

`lag_1h`는 현재 표의 바로 앞 행이 아니라 현재 timestamp에서 정확히 한 시간 전의
target입니다.

| timestamp | `cnt` | `shift(1)` | 정확한 `lag_1h` |
|---|---:|---:|---:|
| 08:00 | 17 | 결측 | 결측 |
| 10:00 | 31 | 17 | 결측 |

09:00 관측이 없으므로 10:00의 앞 행인 08:00은 두 시간 전입니다. timestamp join은
정확한 짝이 없는 lag를 결측으로 남기며, 제외된 행 수와 기간도 기록합니다.


In [ ]:
import pandas as pd
from IPython.display import display

data = pd.DataFrame({
    "timestamp": pd.to_datetime(["2026-01-01 08:00", "2026-01-01 10:00"]),
    "cnt": [17, 31],
}).sort_values("timestamp")
data["previous_row_value"] = data["cnt"].shift(1)
data["required_lag_timestamp"] = data["timestamp"] - pd.Timedelta(hours=1)

history = data[["timestamp", "cnt"]].rename(
    columns={"timestamp": "required_lag_timestamp", "cnt": "exact_lag_1h"}
)
comparison = data.merge(history, on="required_lag_timestamp", how="left")

print("[timestamp로 확인] 앞 행과 정확한 1시간 lag는 같은 개념이 아닙니다.")
display(comparison)
print("10:00의 앞 행은 08:00이지만, 정확한 09:00 관측이 없으므로 exact lag는 결측입니다.")


## Random split과 chronological holdout은 질문이 다르다

| 분할 | 시간 구성 | 평가하는 상황 |
|---|---|---|
| random split | 전체 기간의 행이 양쪽에 섞임 | 같은 기간·분포의 새 행 |
| chronological holdout | 과거는 train, 이후는 test | 과거 자료로 미래기간 예측 |

미래예측 보고에는 시간순 경계가 필요합니다. Tashman (2000)은 out-of-sample 예측을
해석하려면 forecast origin, test period와 재적합 방식을 명시해야 한다고 정리했습니다.

출처: [Tashman (2000)](https://doi.org/10.1016/S0169-2070(00)00065-0)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

dates = pd.date_range("2026-01-01", periods=12, freq="D")
random_test_positions = {1, 5, 9}
split_view = pd.DataFrame({"timestamp": dates})
split_view["random_split"] = ["test" if i in random_test_positions else "train" for i in range(len(dates))]
split_view["chronological_split"] = ["train" if i < 9 else "test" for i in range(len(dates))]

print("[시간범위로 확인] random split은 기간이 섞이고 chronological split은 경계가 분명합니다.")
display(split_view)

palette = {"train": "#2563eb", "test": "#dc2626"}
fig, axes = plt.subplots(2, 1, figsize=(8, 4.2), sharex=True)
for ax, column, title in zip(
    axes,
    ["random_split", "chronological_split"],
    ["Random split", "Chronological holdout"],
):
    for role in ["train", "test"]:
        part = split_view.loc[split_view[column].eq(role)]
        ax.scatter(part["timestamp"], [1] * len(part), color=palette[role], s=55, label=role)
    ax.set(yticks=[], title=title)
axes[0].legend(ncol=2)
axes[-1].set_xlabel("time")
plt.tight_layout()
plt.show()


## Persistence: 직전 이용 가능한 값으로 만든 기준예측

$$\hat y_t^{persistence}=y_{t-h}$$

예측간격 $h$가 15분이면 현재시각보다 정확히 15분 전의 관측값을 예측으로 사용합니다.
후보모형과 같은 미래 timestamp, 같은 horizon과 같은 지표에서 비교해야 합니다.

인접 시점의 값이 비슷한 시계열에서는 persistence가 강한 기준이 될 수 있습니다.
복잡한 모형이 이 기준보다 개선되지 않으면 추가 복잡성을 선택할 근거가 약합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

series = pd.DataFrame({
    "timestamp": pd.date_range("2026-01-01 08:00", periods=4, freq="15min"),
    "actual": [17, 31, 25, 40],
})
series["persistence_prediction"] = series["actual"].shift(1)
series["residual"] = series["actual"] - series["persistence_prediction"]
series["absolute_error"] = series["residual"].abs()
evaluation = series.dropna().copy()

print("[기준예측으로 확인] persistence는 직전 15분의 실제값을 다음 시점 예측으로 사용합니다.")
display(evaluation)
print("persistence MAE:", round(evaluation["absolute_error"].mean(), 2))

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(series["timestamp"], series["actual"], marker="o", label="actual")
ax.plot(series["timestamp"], series["persistence_prediction"], marker="o", linestyle="--", label="persistence")
ax.set(xlabel="timestamp", ylabel="target", title="Persistence uses the last available observation")
ax.legend()
plt.tight_layout()
plt.show()


## 시간축 정보누수와 검증 기록

- 전체 기간의 평균·표준편차로 전처리한 뒤 분할하지 않습니다.
- 미래 target을 포함한 이동평균이나 집계값을 feature로 쓰지 않습니다.
- test 점수를 보며 feature와 설정을 반복 선택하지 않습니다.
- 예측 시점에 없었던 target-time 실제 날씨를 운영 가능한 정보로 간주하지 않습니다.

최종 기록에는 정확한 lag timestamp, train·test 시간 경계, persistence와 후보의 동일
평가행, MAE 단위와 아직 검증하지 않은 계절·기간을 포함합니다.


## 실행 환경과 입력자료

Steel 15분 자료를 시간순으로 정렬한 뒤 다음 항목을 먼저 고정합니다.

| 확인 항목 | 이번 분석의 기준 |
|---|---|
| target | 다음 시점의 `Usage_kWh` |
| 예측 기준시점·간격 | 현재 관측 직후 · 15분 앞 |
| 핵심 lag | 정확히 15분 전 `Usage_kWh` |
| 기준모형 | 직전 관측값을 반복하는 persistence |
| 평가 | 과거 train · 이후 test의 같은 timestamp |

다음 셀에서 파일 경로, 시간범위, lag 생성 전후 행 수와 제외된 행 수를 확인합니다.

> **실행 전 확인:** 첫 예측행의 결과시각과 그 행에서 사용하는 lag의 원래 시각은 각각 언제입니까?


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 정확한 과거 시점의 lag 연결

정렬된 표의 앞 행과 일정 시간 전 관측은 누락 시각이 있을 때 서로 달라집니다.

| 현재 시각 | 연결할 값 | 시간 검증 |
|---|---|---|
| $t$ | `Usage_kWh` at $t-15$분 | 두 timestamp의 차이가 15분 |
| $t$ | `Usage_kWh` at $t-24$시간 | 하루 전 같은 시각 존재 |

과거 자료의 timestamp를 앞으로 이동한 뒤 원자료와 join합니다. 정확한 짝이 없으면 lag는 결측으로 남습니다. 코드 출력에서는 모델자료의 행 수, 전체 시간 범위와 첫 lag 행을 확인합니다.

> **확인:** 09:00 관측이 없을 때 10:00의 바로 앞 행을 `lag_1h`로 사용할 수 없는 이유는 무엇일까요?


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

raw = pd.read_csv(find_course_data("steel/Steel_industry_data.csv"))
raw["timestamp"] = pd.to_datetime(raw["date"], dayfirst=True)
raw = raw.sort_values("timestamp")
target_name = "Usage_kWh"
analysis_name = "Steel one-step time validation"
lag = raw[["timestamp", target_name]].copy()
lag["timestamp"] = lag["timestamp"] + pd.Timedelta(minutes=15)
lag = lag.rename(columns={target_name: "lag_15m"})
data = raw.merge(lag, on="timestamp", how="left").dropna(subset=["lag_15m"]).copy()
data["hour_sin"] = np.sin(2 * np.pi * data["timestamp"].dt.hour / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["timestamp"].dt.hour / 24)
features = ["lag_15m", "hour_sin", "hour_cos"]

print("rows after exact lag join:", len(data))
print("time range:", data["timestamp"].min(), data["timestamp"].max())
display(data[["timestamp", target_name, *features]].head(3))


### 출력 해석: 누락 시각에서는 앞 행과 정확한 과거시각이 다르다

09:00이 빠진 작은 표로 `shift(1)`의 위험을 먼저 확인한 뒤 실제 Steel의 `lag_15m`를 봅니다.


In [ ]:
missing_time_example = pd.DataFrame({
    "timestamp": pd.to_datetime(["2026-01-01 08:00", "2026-01-01 10:00"]),
    "value": [17.0, 31.0],
})
missing_time_example["previous row value"] = missing_time_example["value"].shift(1)
exact_source = missing_time_example[["timestamp", "value"]].copy()
exact_source["timestamp"] = exact_source["timestamp"] + pd.Timedelta(hours=1)
exact_source = exact_source.rename(columns={"value": "exact 1h lag"})
missing_time_example = missing_time_example.merge(exact_source, on="timestamp", how="left")
print("[누락 시각 예시] 10:00의 앞 행은 08:00이지만 정확한 1시간 전 09:00 관측은 없습니다.")
display(missing_time_example)

lag_view = data[["timestamp", target_name, "lag_15m"]].head(8).copy()
lag_view["current_minus_lag"] = lag_view[target_name] - lag_view["lag_15m"]
print("[실제 Steel] 현재 timestamp의 target과 정확히 15분 전 target")
display(lag_view)


## 2. 무작위 분할과 시간순 분할

| 분할 | train/test의 시간관계 | 성능이 적용되는 상황 |
|---|---|---|
| random | 두 집단의 시작·종료기간이 겹침 | 같은 기간·분포의 새로운 행 |
| chronological | train 마지막 시각 < test 첫 시각 | 과거로 이후 기간을 예측 |

두 분할을 같은 코드로 생성한 뒤 시작·종료시각과 행 수를 표로 출력합니다. 미래예측 보고에는 배포 순서를 반영한 chronological 결과를 사용합니다.

> **확인:** 시간순 분할표에서 train의 마지막 시각과 test의 첫 시각은 어떤 부등관계를 가져야 할까요?


In [ ]:
X, y = data[features], data[target_name]
cut = int(len(data) * 0.80)
X_train_t, X_test_t = X.iloc[:cut], X.iloc[cut:]
y_train_t, y_test_t = y.iloc[:cut], y.iloc[cut:]
time_train, time_test = data["timestamp"].iloc[:cut], data["timestamp"].iloc[cut:]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.20, random_state=42)
print("chronological train end:", time_train.max())
print("chronological test start:", time_test.min())


### 출력 해석: 시간순 분할과 무작위 분할

시간순 분할은 경계가 하나지만 무작위 분할은 train과 test의 시간이 섞인다는 점을 비교합니다.


In [ ]:
split_time_view = pd.DataFrame({
    "split": ["chronological train", "chronological test", "random train", "random test"],
    "first timestamp": [time_train.min(), time_test.min(), data.loc[X_train_r.index, "timestamp"].min(), data.loc[X_test_r.index, "timestamp"].min()],
    "last timestamp": [time_train.max(), time_test.max(), data.loc[X_train_r.index, "timestamp"].max(), data.loc[X_test_r.index, "timestamp"].max()],
    "rows": [len(X_train_t), len(X_test_t), len(X_train_r), len(X_test_r)],
})
display(split_time_view)


## 3. Persistence와 선형모형 비교

Steel의 15분 선행 예측에서 persistence는 정확히 15분 전 실제값을 현재 시점의 예측으로 사용합니다.

$$\hat y_t^{\mathrm{persistence}}=y_{t-15\mathrm{min}}$$

| 비교 결과 | 해석 |
|---|---|
| chronological persistence | 미래 test에서 직전값을 반복한 기준오차 |
| chronological linear | 같은 미래 test에서 선형모형의 오차 |
| random linear | 시간 순서를 섞은 별도 평가질문의 오차 |

첫 미래행의 실제값·두 예측·두 절대오차를 직접 확인한 뒤 전체 MAE로 확장합니다.

> **확인:** 선형모형의 chronological MAE가 persistence보다 크다면 어떤 결론이 이 평가로 뒷받침될까요?


In [ ]:
model_time = LinearRegression().fit(X_train_t, y_train_t)
model_random = LinearRegression().fit(X_train_r, y_train_r)
pred_time = model_time.predict(X_test_t)
pred_random = model_random.predict(X_test_r)
persistence = X_test_t["lag_15m"].to_numpy()

result_table = pd.DataFrame([
    {"validation": "chronological", "model": "persistence", "MAE": mean_absolute_error(y_test_t, persistence)},
    {"validation": "chronological", "model": "linear", "MAE": mean_absolute_error(y_test_t, pred_time)},
    {"validation": "random", "model": "linear", "MAE": mean_absolute_error(y_test_r, pred_random)},
])
display(result_table)


### 출력 해석: 첫 예측 구간

같은 미래 행에서 실제값·직전값 baseline·선형모형 예측을 나란히 봅니다.


In [ ]:
prediction_view = pd.DataFrame({
    "timestamp": time_test.iloc[:10].to_numpy(),
    "observed": y_test_t.iloc[:10].to_numpy(),
    "persistence": persistence[:10],
    "linear": pred_time[:10],
})
prediction_view["persistence_abs_error"] = (prediction_view["observed"] - prediction_view["persistence"]).abs()
prediction_view["linear_abs_error"] = (prediction_view["observed"] - prediction_view["linear"]).abs()
print("첫 행에서 persistence는 정확히 15분 전 실제값을 반복합니다.")
display(prediction_view)
first_prediction = prediction_view.iloc[0]
print(
    f"첫 미래행: 실제 {first_prediction['observed']:.2f} kWh, "
    f"persistence {first_prediction['persistence']:.2f} kWh, "
    f"절대오차 {first_prediction['persistence_abs_error']:.2f} kWh"
)
print("같은 미래 test 전체의 MAE")
display(result_table.round(3))


## 4. 시계열 평가 조건

| 검증 항목 | 통과 조건 |
|---|---|
| 시간 순서 | train 종료시각 < test 시작시각 |
| lag | 현재 시점보다 정확히 15분 전 값 |
| feature | 결측과 미래정보 없음 |
| 비교행 | persistence와 후보가 같은 test timestamp 사용 |
| 예측 | test 행 수와 예측 길이 일치 |
| 보고범위 | 15분 horizon과 평가기간 명시 |

한 forecast origin의 통과 여부와 여러 기간에서의 안정성은 서로 다른 검증입니다. 후자는 origin을 옮겨 반복 평가합니다.


In [ ]:
checks = pd.Series({
    "strict time order": time_train.max() < time_test.min(),
    "no missing features": not X.isna().any().any(),
    "prediction length": len(pred_time) == len(y_test_t),
    "three comparison rows": len(result_table) == 3,
    "finite MAE": np.isfinite(result_table["MAE"]).all(),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("시계열 순서 검증: PASS")


## ChatGPT 저장 응답 검증: 예측시점의 정보 가용성 감사하기

각 입력변수가 예측 기준시점에 실제로 알려져 있었는지 표로 점검합니다. 열 이름에 `lag`가 있다는 이유만으로 안전하다고 판단하지 않고 **정확한 timestamp 차이와 자료 생성시점**을 확인합니다.

답변에서 미래정보 가능성이 제기되면 해당 열을 바로 삭제하지 말고, 어떤 시각 정보를 추가로 확인해야 하는지 먼저 기록합니다.


In [ ]:
review_prompt = f"""
다음 시계열 예측실험의 정보 가용성을 점검해 주세요.

- 입력변수: {features}
- 학습 종료시각: {time_train.max()}
- 평가 시작시각: {time_test.min()}
- 결과표:\n{result_table.to_string(index=False)}

입력변수 | 값이 나타내는 시각 | 예측 기준시점에 이용 가능? | 추가 확인 항목 형식의 표를 먼저 작성해 주세요.
이어서 무작위 분할과 시간순 분할의 질문 차이, 직전 값 기준모형과 후보모형의 공정한 비교조건을 설명해 주세요.

제약: 열 이름만 보고 정보 가용성을 확정하지 말고, 확인할 수 없으면 필요한 자료 생성시점과 변경이력을 적어 주세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### ChatGPT 저장 응답을 검증할 때

1. 각 입력의 값 기준시각과 실제 이용 가능시각을 구분합니다.
2. 학습 종료시각이 평가 시작시각보다 앞서는지 실제 timestamp로 확인합니다.
3. 의심되는 열은 생성 규칙을 확인한 뒤 유지·수정·제외를 결정합니다.


## 해석 범위

이번 결과는 하나의 forecast origin에서 이후 15분 값을 예측한 시간순 holdout 성능입니다.

여러 계절과 예측 horizon에서의 안정성, 날씨예보 오차와 운영 중 데이터 분포 변화는 평가하지 않았습니다. 실제 적용 전에는 forecast origin을 이동한 반복검증과 운영자료 점검이 필요합니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

예측 기준시점과 예측간격을 먼저 정하고, 정확한 timestamp lag·시간순 분할·직전 값 기준모형으로 미래정보 누수를 막습니다.

### 적용 순서

1. `07_time_series_validation_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `07_time_series_validation_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- 이 feature는 예측 시점에 실제로 알 수 있는 정보입니까?
- 무작위 분할과 시간순 분할의 연구 질문은 어떻게 다릅니까?
- 미래예측 baseline은 어느 시점의 값만 사용해야 합니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
